# MCH–MFT Track Matching — Training & Testing

Regular (XGBoost Logistic) to score (MCH, MFT candidate) pairs.  
Each MCH track forms a **group** of ~up to~ 5 candidates; the model learns to score true matches high and the rest low
The top-ranked candidate is accepted if its score exceeds a tunable threshold.

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import importlib
import Utils

from hipe4ml.tree_handler import TreeHandler
from sklearn.model_selection import GroupShuffleSplit
from scipy.special import softmax
from scipy.stats import ks_2samp
from sklearn.inspection import permutation_importance
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## 1. Load, Format, Engineer Data

In [ ]:
df_original = Utils.get_dataframe("OO-LHC25i4.root", folder_name="DF_*")


In [ ]:
df =df_original.copy() 
importlib.reload(Utils)

In [ ]:
df = Utils.process_dataframe(df, makedummies=False)

FEATURES = [f for f in df.columns.tolist() if f not in Utils.NON_TRAINING_FEATURES]
FEATURES = [ # Temporary manual list of "good enough" features for ONNX testing which are available for running on o2
 'XMCH',
 'YMCH',
 'PhiMCH',
 'TanlMCH',
 'InvQPtMCH',
 'Chi2MCH',
 'PDCA',
 'Rabs',
 'CXXMCH',
 'CYYMCH',
 'CPhiPhiMCH',
 'CTglTglMCH',
 'C1Pt1PtMCH',
#  'CXYMCH',
#  'CPhiYMCH',
#  'CPhiXMCH',
#  'CTglXMCH',
#  'CTglYMCH',
#  'CTglPhiMCH',
#  'C1PtXMCH',
#  'C1PtYMCH',
#  'C1PtPhiMCH',
#  'C1PtTglMCH',
 'XMFT',
 'YMFT',
 'PhiMFT',
 'TanlMFT',
 'InvQPtMFT',
 'Chi2MFT',
#  'TrackTypeMFT',
 'CXXMFT',
 'CYYMFT',
 'CPhiPhiMFT',
 'CTglTglMFT',
 'C1Pt1PtMFT',
#  'CXYMFT',
#  'CPhiYMFT',
#  'CPhiXMFT',
#  'CTglXMFT',
#  'CTglYMFT',
#  'CTglPhiMFT',
#  'C1PtXMFT',
#  'C1PtYMFT',
#  'C1PtPhiMFT',
#  'C1PtTglMFT',
#  'DCAX',
#  'DCAY',
#  'IsAmbig',
#  'MFTMult',
#  'is_dummy',
 'DeltaX',
 'DeltaY',
 'DeltaPhi',
#  'DeltaTanl',
#  'DeltaR',
#  'RelPtDiff',
#  'SameSign',
#  'PtMCH',
#  'PtMFT',
 'DeltaPt',
#  'PullPt',
#  'PullX',
#  'PullY',
#  'PullR',
#  'PullPhi',
#  'PullTanl',
#  'DeltaDirection',
 ]


TARGET = "IsSignal"

GROUP  = "mchID"

In [ ]:
# ### Downsample the DataFrame to 20% of the original size while maintaining the distribution of 'mchID'
# # 1. Get unique group IDs
# unique_ids = df["mchID"].unique()

# # 2. Sample 20% of IDs
# n_sample = int(0.4 * len(unique_ids))
# sampled_ids = np.random.choice(unique_ids, n_sample, replace=False)

# # 3. Filter rows belonging to those IDs
# df = df[df["mchID"].isin(sampled_ids)]

In [ ]:
df[FEATURES].describe()

In [ ]:
FEATURES

## 2. Sanity Checks

In [ ]:
n_mch_tracks = df["mchID"].nunique()
n_positive = df["IsSignal"].sum()
candidates_per_track = df.groupby("mchID").size()

print(f"MCH tracks:          {n_mch_tracks:,}")
print(f"Total pairs:         {len(df):,}")
print(f"True matches:        {int(n_positive):,} ({100*n_positive/len(df):.1f}%)")
print(f"Candidates per track: min={candidates_per_track.min()}, "
      f"max={candidates_per_track.max()}, "
      f"mean={candidates_per_track.mean():.2f}")

# Tracks with no true match among candidates
tracks_with_match = df.groupby("mchID")["IsSignal"].max()
n_no_match = (tracks_with_match == 0).sum()
print(f"Tracks with no true match in candidates: {n_no_match:,} "
      f"({100*n_no_match/n_mch_tracks:.1f}%)")

## 4. Train / Test Split

Split is done **by MCH track group**, not by row, to avoid data leakage  
(candidates from the same MCH track must not appear in both train and test).

In [ ]:
groups   = df[GROUP].values
splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42) # shuffle the data and split into train/test sets while ensuring all candidates from the same mchID are in the same set
train_idx, test_idx = next(splitter.split(df, groups=groups))

df_train = df.iloc[train_idx].copy()
df_test  = df.iloc[test_idx].copy()

X_train, y_train = df_train[FEATURES], df_train[TARGET]
X_test,  y_test  = df_test[FEATURES],  df_test[TARGET]

print(f"Train: {len(df_train):,} pairs ({df_train[GROUP].nunique():,} MCH tracks)")
print(f"Test:  {len(df_test):,} pairs ({df_test[GROUP].nunique():,} MCH tracks)")
print(f"Train positive rate: {y_train.mean():.3f}")
print(f"Test  positive rate: {y_test.mean():.3f}")

## 5. Train XGBoost tree

In [ ]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos
print(f"scale_pos_weight = {spw:.2f}  (neg={neg:,}, pos={pos:,})")

model = xgb.XGBClassifier(
    objective="binary:logistic",
    n_estimators=250, # cranked down to 10 from 500 to see if we can tweak the export means
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=spw,
    eval_metric="auc",
    early_stopping_rounds=5,
    random_state=42,
    n_jobs=4,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=True,
)

print(f"\nBest iteration: {model.best_iteration}")

# ── Training curve ────────────────────────────────────────────────────────────
results   = model.evals_result()
train_auc = results["validation_0"]["auc"]
test_auc  = results["validation_1"]["auc"]
iterations = range(1, len(train_auc) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left — full training curve
ax = axes[0]
ax.plot(iterations, train_auc, lw=2, color="steelblue", label="Train AUC")
ax.plot(iterations, test_auc,  lw=2, color="tomato",    label="Test AUC")
ax.axvline(model.best_iteration, color="black", linestyle="--", lw=1.5,
           label=f"Best iteration ({model.best_iteration})")
ax.set_xlabel("Iteration")
ax.set_ylabel("AUC")
ax.set_title("Training curve — full")
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)

# Right — zoomed to last 20%
zoom_start = int(0.8 * len(train_auc))
ax = axes[1]
ax.plot(list(iterations)[zoom_start:], train_auc[zoom_start:],
        lw=2, color="steelblue", label="Train AUC")
ax.plot(list(iterations)[zoom_start:], test_auc[zoom_start:],
        lw=2, color="tomato",    label="Test AUC")
ax.axvline(model.best_iteration, color="black", linestyle="--", lw=1.5,
           label=f"Best iteration ({model.best_iteration})")
ax.set_xlabel("Iteration")
ax.set_ylabel("AUC")
ax.set_title("Training curve — zoomed (last 20%)")
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Gap diagnostic ────────────────────────────────────────────────────────────
final_gap = abs(train_auc[-1] - test_auc[-1])
best_gap  = abs(train_auc[model.best_iteration - 1] -
                test_auc[model.best_iteration - 1])
print(f"Train/test AUC gap at best iteration: {best_gap:.6f}")
print(f"Train/test AUC gap at final iteration: {final_gap:.6f}")
print(f"{'⚠ Possible overfitting' if final_gap > 0.01 else '✓ No significant overfitting detected'}")

## 6. Feature Importance

In [ ]:
importances = pd.Series(
    model.feature_importances_, index=FEATURES
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
importances.plot.barh(ax=ax)
ax.set_xlabel("Feature importance (gain)")
ax.set_title("XGBoost feature importances")
plt.tight_layout()
plt.show()

## 7. Permutation feature importance

In [ ]:
def compute_permutation_importance(
    model, 
    X_val, 
    y_val, 
    features,
    n_repeats=10,
    random_state=42,
    n_jobs=5
):
    """
    Compute permutation importance and return as a sorted Series.
    
    Parameters
    ----------
    model : fitted XGBoost model
    X_val : validation feature matrix
    y_val : validation labels
    features : list of feature names
    n_repeats : number of times to permute (default 10)
    random_state : for reproducibility
    n_jobs : parallel jobs (-1 = all cores)
    
    Returns
    -------
    perm_df : DataFrame with importance, std, and percentile
    """
    
    perm_result = permutation_importance(
        model, 
        X_val, 
        y_val,
        n_repeats=n_repeats,
        random_state=random_state,
        n_jobs=n_jobs
    )
    
    perm_df = pd.DataFrame({
        'feature': features,
        'importance': perm_result.importances_mean,
        'std': perm_result.importances_std,
    }).sort_values('importance', ascending=True)
    
    perm_df['importance_percentile'] = perm_df['importance'].rank(pct=True)
    
    return perm_df


def plot_permutation_importance(
    perm_df,
    figsize=(10, 10),
    title="Permutation Importance (Validation Set)",
    show_threshold=None,
    threshold_label=None
):
    """
    Plot permutation importance with error bars.
    
    Parameters
    ----------
    perm_df : DataFrame from compute_permutation_importance
    figsize : figure size
    title : plot title
    show_threshold : optional importance threshold line to plot
    threshold_label : label for threshold line
    
    Returns
    -------
    fig, ax : matplotlib figure and axis
    """
    
    fig, ax = plt.subplots(figsize=figsize)
    
    y_pos = np.arange(len(perm_df))
    ax.barh(
        y_pos, 
        perm_df['importance'],
        xerr=perm_df['std'],
        error_kw={'ecolor': 'gray', 'alpha': 0.5, 'capsize': 3},
        alpha=0.8,
        edgecolor='black',
        linewidth=0.5
    )
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(perm_df['feature'])
    ax.set_xlabel("Permutation Importance (error bars = ±1 std)")
    ax.set_title(title)
    ax.grid(axis='x', alpha=0.3)
    
    if show_threshold is not None:
        ax.axvline(
            show_threshold, 
            color='red', 
            linestyle='--', 
            linewidth=2,
            label=threshold_label or f'Threshold = {show_threshold:.4f}'
        )
        ax.legend()
    
    plt.tight_layout()
    
    return fig, ax


def compare_importances(
    model,
    X_val,
    y_val,
    features,
    figsize=(14, 10),
    n_repeats=10,
    random_state=42,
):
    """
    Compute and plot both native XGBoost importance and permutation importance side-by-side.
    
    Parameters
    ----------
    model : fitted XGBoost model
    X_val : validation features
    y_val : validation labels
    features : list of feature names
    figsize : figure size
    n_repeats : number of permutation repeats
    random_state : for reproducibility
    
    Returns
    -------
    perm_df : DataFrame with permutation importance
    fig, axes : matplotlib figure and axes
    """
    
    # Native importance
    native_imp = pd.Series(
        model.feature_importances_,
        index=features
    ).sort_values(ascending=True)
    
    # Permutation importance
    perm_df = compute_permutation_importance(
        model, X_val, y_val, features,
        n_repeats=n_repeats,
        random_state=random_state
    )
    
    # Create side-by-side plots
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # Native importance (left)
    axes[0].barh(range(len(native_imp)), native_imp.values, alpha=0.8, edgecolor='black', linewidth=0.5)
    axes[0].set_yticks(range(len(native_imp)))
    axes[0].set_yticklabels(native_imp.index)
    axes[0].set_xlabel("Feature Importance (gain)")
    axes[0].set_title("XGBoost Native Importance")
    axes[0].grid(axis='x', alpha=0.3)
    
    # Permutation importance (right)
    y_pos = np.arange(len(perm_df))
    axes[1].barh(
        y_pos,
        perm_df['importance'],
        xerr=perm_df['std'],
        error_kw={'ecolor': 'gray', 'alpha': 0.5, 'capsize': 3},
        alpha=0.8,
        edgecolor='black',
        linewidth=0.5
    )
    axes[1].set_yticks(y_pos)
    axes[1].set_yticklabels(perm_df['feature'])
    axes[1].set_xlabel("Permutation Importance (error bars = ±1 std)")
    axes[1].set_title("Permutation Importance (Validation)")
    axes[1].grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    
    return perm_df, fig, axes


# Example usage:
"""
# Assuming you have: model, X_val, y_val, FEATURES

# Option 1: Just permutation importance
perm_df = compute_permutation_importance(model, X_val, y_val, FEATURES)
fig, ax = plot_permutation_importance(perm_df)
plt.show()

# Option 2: Compare side-by-side with native importance
perm_df, fig, axes = compare_importances(model, X_val, y_val, FEATURES)
plt.show()

# Option 3: Apply filtering
perm_df_filtered = perm_df[perm_df['importance_percentile'] > 0.4]
fig, ax = plot_permutation_importance(
    perm_df_filtered,
    title="Permutation Importance (Top 60%)",
    show_threshold=perm_df[perm_df['importance_percentile'] == 0.4]['importance'].values[0],
    threshold_label="40th percentile cutoff"
)
plt.show()

print(perm_df[['feature', 'importance', 'std', 'importance_percentile']].to_string())
"""


# should this be uuh train or test?
# perm_df = compute_permutation_importance(model, X_test, y_test, FEATURES)
# fig, ax = plot_permutation_importance(perm_df)
# plt.show()

perm_df, fig, axes = compare_importances(model, X_test, y_test, FEATURES)
plt.show()


# TODO: add a plot of the difference between ranking boost & permutation importance - see discrepancy and say smth

In [ ]:
# perm_df = compute_permutation_importance(model, X_test, y_test, FEATURES) unneeded as it is already calculated in the compare_importances function
perm_df_filtered = perm_df[perm_df['importance_percentile'] > 0.4]
fig, ax = plot_permutation_importance(
    perm_df_filtered,
    title="Permutation Importance (Top 60%)",
    show_threshold=perm_df[perm_df['importance_percentile'] >= 0.4]['importance'].values[0],
    threshold_label="40th percentile cutoff"
)
plt.show()

print(perm_df[['feature', 'importance', 'std', 'importance_percentile']].to_string())

In [ ]:


# Get native importance as DataFrame
native_df = pd.DataFrame({
    'feature': FEATURES,
    'native_importance': model.feature_importances_
})

# Prepare permutation DataFrame
perm_df_renamed = perm_df[['feature', 'importance']].rename(columns={'importance': 'perm_importance'})

# Merge the two
importance_comparison = pd.merge(native_df, perm_df_renamed, on='feature')

# Normalize each importance metric to [0,1] by dividing by its maximum
importance_comparison['native_importance_norm'] = importance_comparison['native_importance'] / importance_comparison['native_importance'].max()
importance_comparison['perm_importance_norm'] = importance_comparison['perm_importance'] / importance_comparison['perm_importance'].max()

# Compute difference (normalized permutation - normalized native)
importance_comparison['difference'] = importance_comparison['perm_importance_norm'] - importance_comparison['native_importance_norm']

# Sort by absolute difference for better visualization
importance_comparison = importance_comparison.sort_values('difference', key=abs, ascending=False)

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(importance_comparison['feature'], importance_comparison['difference'],
               color=['red' if x < 0 else 'blue' for x in importance_comparison['difference']],
               alpha=0.7)

ax.set_xlabel('Normalized Difference (Permutation - Native Importance)')
ax.set_title('Normalized Difference between Permutation and Native Feature Importance')
ax.grid(axis='x', alpha=0.3)

# Add value labels on bars
for bar, diff in zip(bars, importance_comparison['difference']):
    width = bar.get_width()
    ax.text(width + (0.01 if width >= 0 else -0.01), bar.get_y() + bar.get_height()/2,
            f'{diff:.3f}', ha='left' if width >= 0 else 'right', va='center', fontsize=8)

plt.tight_layout()
plt.show()

# Also print the comparison table
print("Normalized Feature Importance Comparison:")
print(importance_comparison[['feature', 'native_importance_norm', 'perm_importance_norm', 'difference']].round(4))

## 8. Correlation diversion

In [ ]:
def find_correlation_groups(
    X,
    correlation_threshold=0.75,
    method='complete',
    drop_constant=True,
    verbose=True
):
    """
    Find groups of correlated features using hierarchical clustering.
    
    Parameters
    ----------
    X : feature matrix (DataFrame or array)
    correlation_threshold : features with |correlation| >= this are grouped
    method : linkage method ('complete', 'average', 'single')
    drop_constant : remove zero-variance features before clustering
    verbose : print diagnostics
    
    Returns
    -------
    groups : dict mapping group_id -> list of feature names
    linkage_matrix : for dendrogram plotting
    features : list of features actually used (after filtering)
    corr_matrix : correlation matrix
    dropped_features : list of features removed due to data quality issues
    """
    
    if isinstance(X, pd.DataFrame):
        features = X.columns.tolist()
        X_array = X.values
    else:
        features = [f'feature_{i}' for i in range(X.shape[1])]
        X_array = X
    
    # Check for problematic features
    dropped_features = []
    valid_idx = []
    valid_features = []
    
    for i, feat in enumerate(features):
        col = X_array[:, i]
        
        # Check for constant (zero variance)
        if np.var(col) == 0:
            if drop_constant:
                dropped_features.append((feat, 'zero variance'))
                continue
        
        # Check for NaN
        if np.any(np.isnan(col)):
            dropped_features.append((feat, 'contains NaN'))
            continue
        
        # Check for inf
        if np.any(np.isinf(col)):
            dropped_features.append((feat, 'contains inf'))
            continue
        
        valid_idx.append(i)
        valid_features.append(feat)
    
    if verbose and dropped_features:
        print(f"Dropped {len(dropped_features)} features due to data quality:")
        for feat, reason in dropped_features:
            print(f"  - {feat}: {reason}")
    
    # Subset X to valid features
    X_clean = X_array[:, valid_idx]
    
    # Compute correlation and distance
    corr_matrix = np.corrcoef(X_clean.T)
    
    # Check for NaN in correlation matrix (still might happen with edge cases)
    if np.any(np.isnan(corr_matrix)):
        nan_pairs = np.where(np.isnan(corr_matrix))
        if verbose:
            print(f"Warning: {len(nan_pairs[0])} NaN values in correlation matrix")
        # Replace NaN with 0 (uncorrelated)
        corr_matrix = np.nan_to_num(corr_matrix, nan=0.0)
    
    # Convert correlation to distance (1 - |correlation|)
    distance_matrix = 1 - np.abs(corr_matrix)
    
    # Hierarchical clustering
    linkage_matrix = linkage(
        distance_matrix[np.triu_indices_from(distance_matrix, k=1)],
        method=method
    )
    
    # Cut dendrogram at threshold
    distance_threshold = 1 - correlation_threshold
    cluster_labels = fcluster(linkage_matrix, distance_threshold, criterion='distance')
    
    # Build groups
    groups = {}
    for label, feature in zip(cluster_labels, valid_features):
        if label not in groups:
            groups[label] = []
        groups[label].append(feature)
    
    if verbose:
        print(f"Clustered {len(valid_features)} features into {len(groups)} groups (ρ threshold = {correlation_threshold})")
    
    return groups, linkage_matrix, valid_features, corr_matrix, dropped_features
 
 
def select_from_correlation_groups(
    groups,
    perm_df,
    corr_matrix,
    features,
    method='permutation_importance'
):
    """
    From each correlation group, select the most informative feature.
    
    Parameters
    ----------
    groups : dict from find_correlation_groups
    perm_df : DataFrame from compute_permutation_importance
    corr_matrix : correlation matrix
    features : list of all feature names
    method : 'permutation_importance' (best) or 'ks_stat' if you have KS
    
    Returns
    -------
    selected_features : list of features to keep
    group_info : DataFrame showing which features were dropped from each group
    """
    
    selected = []
    group_info = []
    
    for group_id, group_features in groups.items():
        if len(group_features) == 1:
            # No correlation, keep it
            selected.append(group_features[0])
            continue
        
        # Get importance for features in this group
        group_importance = perm_df[perm_df['feature'].isin(group_features)].copy()
        
        if len(group_importance) == 0:
            # Features not in perm_df (shouldn't happen), keep first
            selected.append(group_features[0])
            continue
        
        # Select best by permutation importance
        best_idx = group_importance['importance'].idxmax()
        best_feature = group_importance.loc[best_idx, 'feature']
        selected.append(best_feature)
        
        # Log what was dropped
        dropped = group_importance[group_importance['feature'] != best_feature]['feature'].tolist()
        for drop_feat in dropped:
            max_corr_in_group = np.abs(
                corr_matrix[
                    features.index(drop_feat),
                    [features.index(f) for f in group_features if f != drop_feat]
                ]
            ).max()
            
            group_info.append({
                'kept_feature': best_feature,
                'dropped_feature': drop_feat,
                'kept_importance': group_importance[group_importance['feature'] == best_feature]['importance'].values[0],
                'dropped_importance': group_importance[group_importance['feature'] == drop_feat]['importance'].values[0],
                'max_correlation_in_group': max_corr_in_group,
                'group_size': len(group_features)
            })
    
    group_info_df = pd.DataFrame(group_info)
    
    return selected, group_info_df
 
 
def plot_correlation_heatmap(
    corr_matrix,
    features,
    selected_features=None,
    figsize=(12, 10),
    threshold=0.75
):
    """
    Plot correlation heatmap with selected features highlighted.
    
    Parameters
    ----------
    corr_matrix : correlation matrix
    features : list of feature names
    selected_features : features to highlight (optional)
    figsize : figure size
    threshold : show threshold line on colorbar
    """
    
    fig, ax = plt.subplots(figsize=figsize)
    
    sns.heatmap(
        np.abs(corr_matrix),  # Use absolute correlation
        xticklabels=features,
        yticklabels=features,
        cmap='RdYlBu_r',
        center=0.5,
        vmin=0,
        vmax=1,
        cbar_kws={'label': '|Correlation|'},
        ax=ax,
        square=True,
        linewidths=0.5
    )
    
    ax.set_title(f'Feature Correlation Matrix (|ρ| threshold = {threshold})')
    plt.tight_layout()
    
    return fig, ax
 
 
# ============================================================================
# EXAMPLE WORKFLOW
# ============================================================================
 

# 1. Find correlated groups (with data quality checks)
groups, linkage_mat, features, corr_matrix, dropped_features = find_correlation_groups(
    X_train.loc[:, (X_train != 0).any(axis=0)],  # Drop all-zero columns
    correlation_threshold=0.75
)
 
print(f"Found {len(groups)} groups")
for gid, feats in groups.items():
    if len(feats) > 1:
        print(f"  Group {gid}: {feats}")
 
# 2. Select best from each group using permutation importance
selected_features, group_info = select_from_correlation_groups(
    groups, 
    perm_df,  # from your permutation importance pipeline
    corr_matrix,
    features
)
 
print(f"\nSelected {len(selected_features)} features from {len(features)} original")
print("\nDropped features:")
print(group_info[['kept_feature', 'dropped_feature', 'kept_importance', 'dropped_importance', 'max_correlation_in_group']])
 
# 3. Visualize
fig, ax = plot_correlation_heatmap(corr_matrix, features, selected_features, threshold=0.75)
plt.show()
 
# 4. Train on reduced feature set
X_filtered = X_train[selected_features]


## 9. Group-Level Evaluation

In [ ]:
df_test = df_test.copy()
df_test["score"] = model.predict_proba(X_test)[:, 1]


METRICS = ["score"]

In [ ]:
# g = xgb.to_graphviz(
#     model,
#     num_trees=10,
#     graph_attr={"dpi": "300", "size": "20,10"}
# )

# g.render("xgb_tree", format="png", cleanup=True)

## 10. All matches

In [ ]:
match_groups = Utils.build_match_groups(df_test)
Utils.draw_all_features(features=METRICS, match_groups=match_groups, density=True)

## 11. Leading Match Metric Distribution Analysis

In [ ]:
df_leader = df_test.loc[df_test.groupby("mchID")["score"].idxmax()].reset_index(drop=True)
match_groups_leader = Utils.build_match_groups(df_leader)
Utils.draw_all_features(features=METRICS, match_groups=match_groups_leader, density=True, per=0.0)

## 12. Metric score sweep

In [ ]:
importlib.reload(Utils)

for entry in METRICS:
    Utils.sweep_threshold_plot(df_eval= df_test, metrics_fn = Utils.inhousemetrics, title=entry +" vs Score Threshold", score_col=entry, Nsigma=3.0)

## 13. Match Assigned Analysis

In [ ]:
# Apply threshold to get final matches, then plot feature distributions for the accepted candidates - see firsthand the contamination
threshold = 0.8
df_leader = df_test.loc[df_test.groupby("mchID")["score"].idxmax()].reset_index(drop=True)
df_leader = df_leader[df_leader["score"] >= threshold].reset_index(drop=True)
match_groups_leader = Utils.build_match_groups(df_leader)
Utils.draw_all_features(features=FEATURES, match_groups=match_groups_leader, density=True)

## 14. Featurewise Metric breakdown

In [ ]:
importlib.reload(Utils)
mch_cols = ["PhiMCH", "TanlMCH", "InvQPtMCH", "PtMCH"] # MCH cols we can actually use for these metrics as they require the underlying group structure to be preserved
common_cols= mch_cols + ['PDCA', 'Rabs', 'MFTMult']
#TODO: confirm these preserve grouping. Add non mch group preserving structures... like possibly MFTMult if it remains defined based on the best chi2 tracks around a mch track - correspond t
for entry in common_cols:   
    Utils.plot_metrics_vs_feature(df=df_test,feature=entry, threshold = 0.8, metrics_fn=Utils.inhousemetrics, metric_col_prefix="score", bins=25, trim_low=0.05, trim_high=0.005, Nsigma=1.0)

## 15. Feature Selection

In [ ]:
# ── Thresholds — adjust these ─────────────────────────────────────────────────
KS_THRESHOLD         = 0.1
IMPORTANCE_THRESHOLD = 0.01

# ── Compute KS statistic for each feature ────────────────────────────────────
signal     = df[df["IsSignal"] == 1]
background = df[df["IsSignal"] == 0]

ks_stats = {
    feature: ks_2samp(signal[feature].dropna(),
                      background[feature].dropna()).statistic
    for feature in FEATURES
}

importance = dict(zip(FEATURES, model.feature_importances_))

selection_df = pd.DataFrame({
    "ks":         ks_stats,
    "importance": importance,
}).sort_values("ks", ascending=False)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

passes = selection_df[
    (selection_df["ks"]         >= KS_THRESHOLD) &
    (selection_df["importance"] >= IMPORTANCE_THRESHOLD)
]
fails = selection_df[
    (selection_df["ks"]         <  KS_THRESHOLD) |
    (selection_df["importance"] <  IMPORTANCE_THRESHOLD)
]

ax.scatter(fails["ks"],   fails["importance"],
           alpha=0.6, color="tomato",    s=60, label="Rejected")
ax.scatter(passes["ks"],  passes["importance"],
           alpha=0.8, color="steelblue", s=60, label="Selected")

for feature, row in selection_df.iterrows():
    ax.annotate(feature, (row["ks"], row["importance"]),
                fontsize=8, textcoords="offset points", xytext=(5, 3))

ax.axvline(KS_THRESHOLD,         color="black", linestyle="--", lw=1.2,
           label=f"KS threshold ({KS_THRESHOLD})")
ax.axhline(IMPORTANCE_THRESHOLD, color="grey",  linestyle="--", lw=1.2,
           label=f"Importance threshold ({IMPORTANCE_THRESHOLD})")

ax.set_xlabel("KS statistic (univariate separability)")
ax.set_ylabel("Feature importance (model usage)")
ax.set_title("Feature selection overview")
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ── Print selected features ───────────────────────────────────────────────────
selected = passes.index.tolist()
dropped  = fails.index.tolist()

print(f"Selected ({len(selected)}) — copy-paste ready:")
print("FEATURES = [")
for f in selected:
    print(f"    \"{f}\",")
print("]")

print(f"\nDropped ({len(dropped)}):")
print(dropped)
print(selection_df.round(4))

## 16. ONNX

In [ ]:
import sklearn

In [ ]:
model_original = sklearn.base.clone(model)

In [ ]:

model.get_booster().feature_names = [f"f{i}" for i in range(len(FEATURES))]

In [ ]:
import onnx
import onnxmltools
from onnxmltools.convert.common.data_types import FloatTensorType
import onnxruntime as ort



In [ ]:
print(xgb.__version__)
print(onnx.__version__)

In [ ]:
initial_types = [('float_input', FloatTensorType([-1, len(FEATURES)]))]
onnx_model = onnxmltools.convert_xgboost(model, initial_types=initial_types)
onnxmltools.utils.save_model(onnx_model, 'model_newtest.onnx')

In [ ]:
X_test_numpy = df[FEATURES].to_numpy(dtype=np.float32)

In [ ]:
selectedinput = np.array([[
    5.282344
,   -3.9980261
,   0.9145084
,   -24.896418
,   1.5657754
,   0.29101562
,   94.99072
,   18.571564
,   18.073513
,   18.158918
,   0.07550935
,   46.82407
,   0.185986
,   2.798654
,   3.1521912
,   0.8196365
,   -19.970703
,   14.083008
,   1.3359375
,   2.7854223e-05
,   2.3339007e-05
,   4.8247442e-05
,   0.0043173116
,   10.145764
,   2.4836898
,   -7.150217
,   0.09487186
,   0.56765366
]], dtype=np.float32)
expectedscore = 0.62250906

In [ ]:
xgb_pred = model.predict_proba(X_test_numpy)[:, 1]
print(f"XGBoost prediction: {xgb_pred}") 

In [ ]:
sess = ort.InferenceSession("model.onnx")
input_name = sess.get_inputs()[0].name
onnx_label, onnx_prob = sess.run(None, {input_name: X_test_numpy})
print(onnx_prob)

In [ ]:
for o in sess.get_outputs():
    print(o.name, o.shape, o.type)

In [ ]:
print(sess.get_inputs()[0])

In [ ]:
diff = np.abs(xgb_pred - onnx_prob[:, 1])  # Assuming onnx_pred is a list of outputs and the probabilities are in the first output
print("Max abs diff:", np.max(diff))
print("Mean abs diff:", np.mean(diff))